In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py

from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","1_Domain_Profiles"))
from CLASSES_DomainProfiles import DomainProfiles_Class, DomainProfiles_DataLoading_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Eulerian_CLTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=20
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=100
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#MODEL AND ALGORITHM NUMERICAL PARAMETERS

In [ ]:
kms=np.argmax(ModelData.xh-ModelData.xh[0] >= 1) #finds how many x grids is 1 km
dx=int(ModelData.xh[1]-ModelData.xh[0]) #grid resolution (in km)

In [ ]:
#setting convergence threshold
if ModelData.res=='1km':
    conv_thresh=1.0/1000
elif ModelData.res=="250m":
    conv_thresh=1.5/1000

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def GetConvergence(t):
    timeString = ModelData.timeStrings[t]
    varName = 'convergence'
    convergence = CallVariable(ModelData, DataManager, timeString, varName)
    return convergence

In [ ]:
##############################################
#ALGORITHM FUNCTIONS

In [ ]:
#SBZ Convergence Line Search Algorithm (levels are seperate) (python version 3.10.9) (All Max Algorithm)
def FindMaxIndices(yconv):
    """find indices of local maxima above threshold."""

     #takes dconv/dx
    ddx = (yconv[1:  ] - yconv[0:-1]) / (2 * dx)
     
    #finds local max where dconv/dx sign changes
    signs = np.sign(ddx)
    signs_diff=np.diff(signs)
    local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here (it corrects the location of the derivative)
    local_maxes=local_maxes[np.where(yconv[local_maxes]>conv_thresh)] #check if convergence is greater than convergence threshold (1s-1)
    local_maxes=local_maxes[(local_maxes>50*kms)&(local_maxes<len(ModelData.xf)-50*kms)] #removes maxes that are with 50 km of y boundary
    return local_maxes

In [ ]:
#Finds all local maximums (from Calculus) along each y level for a specific z level (~0.28km in this case)
def FindLocalMaxes(conv_z,yind, second_round=False):
    """
    #Find_Local_Maxes Function
    #(1) At a single time and z level, runs through each y-level
    #(2) At each y-level, takes the x-derivative
    #(3) Take sign(x_derivative)
    #(4) Take diff(x_derivative)
    #(5) Max is located one index to the right of where derivative changes from positive to negative or diff is +1
    #[(6) Optional: the algorithm can run a second time over the leftover maxes after removing previous maxes from temporary variable]
    """
    
    #indexes convergence in y
    yconv=conv_z[yind,:]
    
    #finds local max where dconv/dx sign changes
    local_maxes = FindMaxIndices(yconv)

    #second round maxes (not 100% necessary, only if missing many convergence maximums that are visually there)
    if second_round == True:
        yconv2=yconv.copy()
        yconv2[local_maxes]=0
        
        #takes dconv/dx
        local_maxes2 = FindMaxIndicies(f=yconv2)
    
        #combine the two
        local_maxes=np.concatenate((local_maxes,local_maxes2))
    return local_maxes

In [ ]:
##############################################
#RUNNING FUNCTIONS

In [ ]:
def LayerMax(conv): #finds max convergence along y for multiple z location (5 is good)
    
    #initializing data
    Nz = int(np.where(ModelData.zh <= 0.775)[0][-1]) + 1 #only include levels less than 1 km
    maxConvergence_X=np.full((Nz, ModelData.Nyh, ModelData.Nxh), -1, dtype=np.int16)
    
    #running for all z levels
    for zlev in range(Nz):
        #Taking Convergence of current timesftep
        conv_z=conv[zlev,:,:] #current z level for convergence

        for yind in range(ModelData.Nyh): #plot maximums for each row

            #finds all local maxes
            local_maxes=FindLocalMaxes(conv_z,yind) #convergence threshold (in 1/s)
            
            #storing data
            maxConvergence_X[zlev,yind,local_maxes] = local_maxes
    return maxConvergence_X

def GetAvgConvergence(conv):
    # #OPTION ONE
    # z_level = np.where(ModelData.zh<=1)[0][-1]
    # return np.mean(conv[0:z_level+1], axis=(1)) #worst option
    
    # #OPTION TWO
    # z_level = np.where(ModelData.zh<=500/1e3)[0][-1]
    # return np.mean(conv[z_level],axis=0) #second best option
    
    #OPTION THREE
    return np.mean(conv, axis=(0,1)) #BEST OPTION

In [ ]:
def RunAlgorithm():
    for t in tqdm(loop_elements, desc="Processing timesteps"):
        # if t % 1 == 0: print(f'Processing timestep {t}/{len(data["time"])}')

        #getting convergence at time t
        conv=GetConvergence(t)

        # Compute maxConvergence_X for this timestep (z,y,x)
        maxConvergence_X = LayerMax(conv)

        avgConvergence = GetAvgConvergence(conv)

        Dictionary = {"maxConvergence_X": maxConvergence_X,
                      "avgConvergence": avgConvergence}

        # Directly write into HDF5 dataset at index t
        timeString = ModelData.timeStrings[t]
        TrackingAlgorithms_DataLoading_Class.SaveData(ModelData,DataManager, Dictionary, timeString)

In [ ]:
#############################################
#RUNNING

In [ ]:
# RunAlgorithm()

In [ ]:
#############################################
#TESTING

In [ ]:
#Testing Different Convergence Thresholds

#import classes
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS
from datetime import datetime
def GetTimesFromTimeIndiceRanges(timeIndiceRange):
    split=timeIndiceRange.split("-")
    [a,b] = int(split[0].split('_')[1]), int(split[1])
    return a,b

def ConvertTimeString(tString, initialTime=6):

    # parse string (HH-MM-SS)
    dt = datetime.strptime(tString, "%H-%M-%S")

    # add hours
    dt = dt + timedelta(hours=initialTime)

    # format output
    return dt.strftime("%H:%M")
    
def GetTimeInfo(timeIndiceRange):    
    [a,b] = GetTimesFromTimeIndiceRanges(timeIndiceRange)
    timeStrings_datetime = TrackedProfiles_DataLoading_CLASS.ConvertTimeStringsToDatetime(ModelData.timeStrings)
    tIndexes = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,a,b)
    tIndex = tIndexes[len(tIndexes)//2]
    tString = ConvertTimeString(ModelData.timeStrings[tIndex])
    return tIndex,tString 

def GetResult(t=100):
    zIndex=(np.abs(ModelData.zh - zHeight)).argmin()
    conv=GetConvergence(t); maxConvergence_X = LayerMax(conv)[zIndex]
    conv = conv[zIndex]
    return conv,maxConvergence_X

def MakePlot_conv(conv,
             xh=ModelData.xh-(ModelData.xh[0]),
             yh=ModelData.yh-(ModelData.yh[0]),
             axis=None, cmap='RdBu_r'):
    if axis is None:
        plt.figure(figsize=(10, 4))
        plt.pcolormesh(xh,yh,conv, cmap=cmap)
        plt.xlim(left=75);plt.xlabel('x (km)');plt.ylabel('y (km)')
    else:
        axis.pcolormesh(xh,yh,conv, cmap=cmap)
        axis.set_xlim(left=75);axis.set_xlabel('x (km)');axis.set_ylabel('y (km)')
def MakeAllPlots_conv():

    fig, axes = plt.subplots(3, 2, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4)
    axes = axes.flatten()  # makes indexing easier
    
    for i, timeIndiceRange in enumerate(timeIndiceRanges):
        axis = axes[i]
        tIndex, tString = GetTimeInfo(timeIndiceRange)
        conv,_ = GetResult(t=tIndex)
    
        MakePlot_conv(conv=conv,
                 axis=axis)
        axis.set_title(f't = {tString}')

    fig.suptitle(f"CLs at {zHeight} km with convergence threshold = {conv_thresh:.1e}",y=0.94)
    return fig

def MakePlot_CL(maxConvergence_X,
         xh=ModelData.xh-(ModelData.xh[0]),
         yh=ModelData.yh-(ModelData.yh[0]),
            axis=None):
    
    cmap = mcolors.ListedColormap(['black']); cmap.set_under('white')

    if axis is None:
        plt.figure(figsize=(10, 4))
        plt.pcolormesh(xh,yh,maxConvergence_X, cmap=cmap, vmin=0)
        plt.xlim(left=75);plt.xlabel('x (km)');plt.ylabel('y (km)')
    else:
        axis.pcolormesh(xh,yh,maxConvergence_X, cmap=cmap, vmin=0)
        axis.set_xlim(left=75);axis.set_xlabel('x (km)');axis.set_ylabel('y (km)')
def MakeAllPlots_CL():

    fig, axes = plt.subplots(3, 2, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4)
    axes = axes.flatten()  # makes indexing easier
    
    for i, timeIndiceRange in enumerate(timeIndiceRanges):
        axis = axes[i]
        tIndex, tString = GetTimeInfo(timeIndiceRange)
        _,maxConvergence_X = GetResult(t=tIndex)
    
        MakePlot_CL(maxConvergence_X=maxConvergence_X,
                 axis=axis)
        axis.set_title(f't = {tString}')

    fig.suptitle(f"CLs at {zHeight} km with convergence threshold = {conv_thresh:.1e}",y=0.94)
    return fig

def PlotCLPoints(xh,yh,maxConvergence_X, axis=None,
                 color='black', size=0.2):
    if axis is None:
        fig, axis = plt.subplots(figsize=(10, 4))

    for yind in range(len(yh)):
        localMaxes = maxConvergence_X[yind]

        # remove invalid points
        localMaxes = localMaxes[localMaxes != -1]

        if len(localMaxes) == 0:
            continue

        localMaxes = localMaxes.astype(int)

        axis.scatter(
            xh[localMaxes],
            # [yh[yind]] * len(local_maxes) #original
            np.full(len(localMaxes), yh[yind]),
            color=color,s=size,zorder=10,alpha=0.7)

    return axis
def MakePlot_combined(conv,maxConvergence_X,
                      theta_v_prime,
                      xh=ModelData.xh-(ModelData.xh[0]),
                      yh=ModelData.yh-(ModelData.yh[0]),
                      axis=None,
                      variableName='conv'):
    if variableName=='conv':
        data=conv
        levels=16
        cmap='RdBu_r'
        norm=None
        color='black'
    elif variableName=="theta_v_prime":
        data=theta_v_prime
        levels = np.arange(-2, 2, 0.25)
        cmap = plt.get_cmap('RdBu_r').copy()
        norm = mcolors.BoundaryNorm(levels, cmap.N)
        color='green'
    
    if axis is None:
        plt.figure(figsize=(10, 4))
        plt.pcolormesh(xh,yh,data, cmap=cmap,norm=norm);plt.colorbar()
        plt.xlim(left=75);plt.xlabel('x (km)');plt.ylabel('y (km)')
    else:
        pcm=axis.pcolormesh(xh,yh,data, cmap=cmap,norm=norm);plt.colorbar(pcm, ax=axis)
        PlotCLPoints(xh,yh,maxConvergence_X,axis=axis,color=color);
        axis.set_xlim(left=75);axis.set_xlabel('x (km)');axis.set_ylabel('y (km)')
def GetTheta_v_prime(t):
    timeString=ModelData.timeStrings[t]
    varName = 'theta_v'
    theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
    theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs[t]+10:])
    return theta_v_prime
def MakeAllPlots_combined(variableName='conv'):

    fig, axes = plt.subplots(3, 2, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4)
    axes = axes.flatten()  # makes indexing easier
    
    for i, timeIndiceRange in enumerate(timeIndiceRanges):
        axis = axes[i]
        tIndex, tString = GetTimeInfo(timeIndiceRange)
        conv,maxConvergence_X = GetResult(t=tIndex)
        theta_v_prime=GetTheta_v_prime(t=tIndex)
    
        MakePlot_combined(conv=conv,maxConvergence_X=maxConvergence_X,
                          theta_v_prime=theta_v_prime,
                          axis=axis,variableName=variableName)
        axis.set_title(f't = {tString}')

    fig.suptitle(f"CLs at {zHeight} km with convergence threshold = {conv_thresh:.1e}",y=0.94)
    return fig
    
# timeIndiceRanges = ['_10-11', '_11-12', '_12-13', '_13-14','_14-15','_15-16']
# zHeight=350/1e3

# # if ModelData.Nzh<40:
# conv_thresh = 1.0/1e3
# fig=MakeAllPlots_combined()

# conv_thresh = 1.5/1e3
# fig=MakeAllPlots_combined()

# # if ModelData.Nzh>80:
# #     conv_thresh = 2.0/1e3
# #     fig=MakeAllPlots_combined()

In [ ]:
# #WHAT IF WE LOOK +=1 or 2km around THETA_V'THRESHOLD to check for cold pools

# t=90
# timeString=ModelData.timeStrings[t]
# varName = 'qr'
# qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
# varName = 'theta_v'
# theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
# # theta_v_prime = theta_v - np.mean(theta_v)
# theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs[t]+10])

# xh=ModelData.xh-(ModelData.xh[0])
# yh=ModelData.yh-(ModelData.yh[0])

# def CL_Locations(t=100):
#     zIndex=(np.abs(ModelData.zh - zHeight)).argmin()
#     conv=GetConvergence(t); maxConvergence_X = LayerMax(conv)[zIndex]
#     maxConvergence_X = LayerMax(conv)[zIndex]
#     return maxConvergence_X

# maxConvergence_X = CL_Locations(t)

# def PlotMask(mask,maxConvergence_X):
#     fig,axis=plt.subplots()
#     cf=axis.contourf(mask, cmap='gray_r')
#     cbar = fig.colorbar(cf, ax=axis)
#     PlotCLPoints(xh,yh,maxConvergence_X,axis=axis,color="red");

# #test plotting
# mask1 = theta_v_prime<-1
# PlotMask(mask1,maxConvergence_X)

# mask2 = qr>1e-6
# PlotMask(mask2,maxConvergence_X)

# # fixed_maxConv = maxConvergence_X.copy()
# # fixed_maxConv[~(mask1&mask2)]=-1
# PlotMask(mask1&mask2,maxConvergence_X)

# radius = int(1*1000 / ModelData.dx)
# # mask_expanded = ApplyMaximumFilter(mask,numKilometers=2) #too large of a distance, use 1 km instead
# mask_expanded = ApplyMaximumFilter(mask1,numKilometers=1)&mask2 #apply a multi-dimensional maximum filter to dilate a binary operation (e.g. mask)
# fixed_maxConv = maxConvergence_X.copy()
# fixed_maxConv[~mask_expanded]=-1
# PlotMask(mask_expanded,fixed_maxConv)

In [ ]:
# def GetTestMasks(t):
#     [_,condition1, _,condition2,SBF_threshold] = GetDataAndMasks(t)

#     a1=~(condition1&condition2)
#     a2=~(ApplyMaximumFilter(condition1,numKilometers=2)&condition2)
#     b=~SBF_threshold
#     return a1,a2,b

# def QuickPlot(axis, theta_v_prime):    
#     data=theta_v_prime.copy()
#     levels = np.arange(-2, 2, 0.25)
#     cmap = plt.get_cmap('RdBu_r').copy()
#     norm = mcolors.BoundaryNorm(levels, cmap.N)
#     pcm=axis.pcolormesh(xh,yh,data, cmap=cmap,norm=norm);plt.colorbar(pcm, ax=axis)
#     axis.set_xlim(left=75);axis.set_xlabel('x (km)');axis.set_ylabel('y (km)')

# step=5 if ModelData.Nzh>80 else 1
# t=90*step
# theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
# # theta_v_prime = theta_v - np.mean(theta_v)
# theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs[t]+10])
# a1,a2,b=GetTestMasks(t)

# fig, AxisArray = plt.subplots(3, 1, figsize=(10, 10))

# # --- Plot 1: No Threshold ---
# data = theta_v_prime.copy()
# QuickPlot(AxisArray[0], data)
# AxisArray[0].set_title("No Threshold")

# # --- Plot 2: theta_v' + qr threshold + 10km SBF ---
# data = theta_v_prime.copy()
# data[a1 & b] = np.nan
# QuickPlot(AxisArray[1], data)
# AxisArray[1].set_title("theta_v' and qr threshold and 10km SBF region")

# # --- Plot 3: max filter + qr threshold + 10km SBF ---
# data = theta_v_prime.copy()
# data[a2 & b] = np.nan
# QuickPlot(AxisArray[2], data)
# AxisArray[2].set_title("maximumfilter(theta_v',2km) and qr threshold and 10km SBF region")

# plt.tight_layout()
# plt.show()

In [ ]:
# def TestPlot_theta_threshold():
#     varName = 'theta_v'
#     theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
#     theta_v_prime = theta_v - np.mean(theta_v)

#     theta_v_prime_threshold = theta_v_prime.copy()
#     condition1 = theta_v_prime_threshold > -1
#     theta_v_prime_threshold[np.where(condition1)] = np.nan
    
#     cmap = plt.get_cmap('RdBu_r'); levels = np.linspace(-2, 2, 10)
#     # plt.contourf(theta_v_prime_threshold,cmap=cmap,levels=levels); plt.colorbar()
#     norm = mcolors.BoundaryNorm(levels, cmap.N)
#     plt.pcolormesh(theta_v_prime_threshold, cmap=cmap, norm=norm)
#     plt.colorbar()

# def TestPlot_qr_threshold():
#     varName = 'qr'
#     qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
#     condition2= qr < 1e-6
#     qr[condition2] = np.nan
#     plt.contourf(qr,cmap='turbo',levels=40); plt.colorbar()

# # tIndex, tString = GetTimeInfo(timeIndiceRanges[5])
# # timeString=ModelData.timeStrings[tIndex]
# # TestPlot_theta_threshold()
# # TestPlot_qr_threshold()

#getting convergence xmax
def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs
xmaxs=find_SBF_xmaxs()

def GetSBFThreshold(t):
    xmaxs_t = xmaxs[t]
    kms = np.argmax(ModelData.xh - ModelData.xh[0] >= 1)
    x = np.arange(len(ModelData.xh))
    y = np.arange(len(ModelData.yh))
    X, Y = np.meshgrid(x, y)
    X = X.astype(float)
    SBF1 = X > xmaxs_t - 10*kms
    SBF2 = X < xmaxs_t + 10*kms
    SBF_threshold=SBF1&SBF2
    return X,SBF_threshold
# [X,SBF_threshold] = GetSBFThreshold(t)
# X[SBF1 & SBF2] = np.nan
# plt.contourf(X,levels=np.max(x));plt.colorbar()

def ApplyMaximumFilter(mask,
                       numKilometers = 2):
    radius = int(np.ceil(numKilometers*1000 / ModelData.dx))
    mask_expanded = maximum_filter(mask, size=2*radius+1)
    return mask_expanded
from scipy.ndimage import maximum_filter

def GetDataAndMasks(t):
    timeString=ModelData.timeStrings[t]
    varName = 'theta_v'
    theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
    # theta_v_prime = theta_v - np.mean(theta_v)
    theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs[t]+10])
    # condition1 = theta_v_prime < -1
    condition1 = theta_v_prime < -1 #50 #==> no theta_v_prime threshold

    varName = 'qr'
    qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
    condition2= qr > 1e-6 #-50 #==> no rain threshold
    
    _,SBF_threshold = GetSBFThreshold(t) 
    return theta_v_prime,condition1, qr,condition2, SBF_threshold
    
# def ApplyThreshold(data,t):
#     [_,condition1, _,condition2,SBF_threshold] = GetDataAndMasks(t)
#     # data[(condition1|condition2)&(~SBF_threshold)] = np.nan
#     data[(condition1&condition2)&(~SBF_threshold)] = np.nan #*testing
#     return data

# def ApplyThreshold_maxConvergence_X(data,t):
#     [_,condition1, _,condition2,SBF_threshold] = GetDataAndMasks(t)
#     # data[~(condition1|condition2) & (~SBF_threshold)] = -1
#     data[~(condition1&condition2) & (~SBF_threshold)] = -1 #*testing
#     return data
def ApplyThreshold_maxConvergence_X(data,t):
    [_,condition1, _,condition2,SBF_threshold] = GetDataAndMasks(t)
    # data[~(condition1|condition2) & (~SBF_threshold)] = -1
    data[~(ApplyMaximumFilter(condition1)&condition2) & (~SBF_threshold)] = -1 #*testing
    return data

# def PlotTheta_v_afterthreshold(data,
#                                cmap='Blues_r'):
#     data[condition1|condition2] = np.nan
    
#     cmap = plt.get_cmap(cmap); levels = np.linspace(-2, -1, 20)
#     cmap.set_bad(color='black')
#     # plt.contourf(data,cmap=cmap,levels=levels); plt.colorbar()
#     norm = mcolors.BoundaryNorm(levels, cmap.N)
#     plt.pcolormesh(data, cmap=cmap, norm=norm)
#     plt.colorbar()

# tIndex, tString = GetTimeInfo(timeIndiceRanges[5])
# [theta_v_prime,condition1, qr,condition2] = GetDataAndMasks(tIndex)
# PlotTheta_v_afterthreshold(data=theta_v_prime)

#Testing Including Cold Pool Threshold Right of SBF+10km
def GetResult(t=100):
    zIndex=(np.abs(ModelData.zh - zHeight)).argmin()
    conv=GetConvergence(t); maxConvergence_X = LayerMax(conv)[zIndex]
    conv = conv[zIndex]
    # return conv,maxConvergence_X
    return conv,ApplyThreshold_maxConvergence_X(maxConvergence_X,t)

timeIndiceRanges = ['_10-11', '_11-12', '_12-13', '_13-14','_14-15','_15-16']
zHeight=350/1e3
variableName="conv"
variableName="theta_v_prime"

# if ModelData.Nzh<40:
conv_thresh = 1.0/1e3
fig=MakeAllPlots_combined(variableName=variableName)

# if ModelData.Nzh>80:
#     conv_thresh = 1.5/1e3
#     fig=MakeAllPlots_combined(variableName=variableName)

In [ ]:
def MakeAllPlots_combined(variableName='conv'):

    fig, axes = plt.subplots(3, 2, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4)
    axes = axes.flatten()  # makes indexing easier
    
    for i, timeIndiceRange in enumerate(timeIndiceRanges):
        axis = axes[i]
        tIndex, tString = GetTimeInfo(timeIndiceRange)
        conv,maxConvergence_X = GetResult(t=tIndex)
        theta_v_prime=GetTheta_v_prime(t=tIndex)
    
        MakePlot_combined(conv=conv,maxConvergence_X=maxConvergence_X,
                          theta_v_prime=theta_v_prime,
                          axis=axis,variableName=variableName)
        axis.set_title(f't = {tString}')

    fig.suptitle(f"CLs at {zHeight} km with convergence threshold = {conv_thresh:.1e}",y=0.94)
    return fig

In [ ]:
#TESTING FINAL VERSION TO ADD TO Eulerian_CLTracking

def GetSBFMask(t):
    xmaxs_t = xmaxs[t]
    kms = np.argmax(ModelData.xh - ModelData.xh[0] >= 1)
    x = np.arange(len(ModelData.xh))
    y = np.arange(len(ModelData.yh))
    X, Y = np.meshgrid(x, y)
    X = X.astype(float)
    SBF1 = X > xmaxs_t - 10*kms
    SBF2 = X < xmaxs_t + 10*kms
    SBF_mask=SBF1&SBF2
    return SBF_mask

def GetDataAndMasks(t):
    timeString=ModelData.timeStrings[t]
    varName = 'theta_v'
    theta_v = CallVariable(ModelData, DataManager, timeString, varName)[0]
    # theta_v_prime = theta_v - np.mean(theta_v) #original
    theta_v_prime = theta_v - np.mean(theta_v[:,xmaxs[t]+10]) #using mean over right of SBF to improve estimate
    theta_v_prime_mask = theta_v_prime < -1

    varName = 'qr'
    qr = CallVariable(ModelData, DataManager, timeString, varName)[0]
    qr_mask= qr > 1e-6 
    
    SBF_mask = GetSBFMask(t) 
    return theta_v_prime,theta_v_prime_mask, qr,qr_mask, SBF_mask
 
def ApplyMaximumFilter(mask,
                       numKilometers = 2):
    radius = int(np.ceil(numKilometers*1000 / ModelData.dx))
    mask_expanded = maximum_filter(mask, size=2*radius+1)
    return mask_expanded
from scipy.ndimage import maximum_filter

def CombineMasks(theta_v_prime_mask,qr_mask,SBF_threshold):
    one=ApplyMaximumFilter(theta_v_prime_mask)
    two=qr_mask
    three=SBF_threshold
    
    CP_SBF_Mask = (one&two) | three
    return CP_SBF_Mask

# for t in tqdm(range(ModelData.Ntime)):
[theta_v_prime,theta_v_prime_mask, qr,qr_mask, SBF_mask] = GetDataAndMasks(t)
CP_SBF_Mask = CombineMasks(theta_v_prime_mask,qr_mask,SBF_mask)